In [2]:
!pip install python-dotenv

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

In [16]:
from openai import OpenAI
# client = OpenAI(api_key = os.environ['OPENAI_KEY'])
client = OpenAI()
def ask_llm(prompt, model = 'gpt-5-nano', temp=1):
    response = client.chat.completions.create(
        model=model,    
        temperature=temp,
        messages=[{
            'role':'user',
            'content' : prompt
        }] 
    )       
    return response.choices[0].message.content   

In [ ]:
# 1. zero-shot Prompting
# 예시없이 지시사항만 던지는것 - LLM의 기본기능
prompt = "이 문장의 감정을 분류해: '오늘 점심 메뉴가 품절이라 너무 슬퍼.'"
print(ask_llm(prompt,temp=1))

주요 감정: 부정적  
세부 감정: 슬픔(강한 수준), 실망도 포함될 수 있습니다.


In [ ]:
# 2. few-shot Prompting
# 이렇게 하는거야 라고 예시(Shot)를 몇개 보여줘서 성능을 높이는 기술
prompt = """
단어를 이모지로 바꿔줘
사과 -> 🍎
자동차 -> 🚗
고양이 -> 🐱
비행기->
"""
print(ask_llm(prompt,temp=1))

비행기 -> ✈️


In [ ]:
# 3. Chain-of-Thought Prompting  Cot(생각의 사슬)
prompt = """
질문 : 5개의 사과중 2개를 먹고 3개를 더 샀어. 몇개 남았지
사과의 단가는 100원
지급한 금액 : 1000원
총 남은 사과의 개수를 세고 그리고 사과를 구입할때 드는 비용을 
계산해서 거스름돈을 계산해줘
계산은 먹은 사과와 남은 사과를 모두 포함한 금액
마지막 출력은 전체 로직을 점검해서 오류가 있는지 확인하고 결과알려줘
"""
print(ask_llm(prompt,temp=1))

다음과 같이 계산해볼게요. 주어진 값은
- 처음 사과: 5개
- 사과를 먹음: 2개
- 남은 사과(먹고 남긴 후) = 5 - 2 = 3개
- 새로 산 사과: 3개
- 산 사과의 단가: 100원
- 지급한 금액: 1000원

1) 현재 남아 있는 사과의 총 개수
- 먹은 뒤 남은 사과 3개 + 새로 산 사과 3개 = 6개

2) 구입에 드는 비용(요청대로 “먹은 사과와 남은 사과를 모두 포함”한 금액으로 계산)
- 총 포함 사과 수 = 먹은 사과 2개 + 현재 남은 사과 6개 = 8개
- 8개 × 100원 = 800원

3) 거스름돈
- 지급액 1000원 − 총 포함 비용 800원 = 200원

추가로 명확히 해두면 좋은 점
- 일반적으로는 현재 가진 사과(6개) 만 계산해서 거스름돈을 구합니다.
  - 이 경우 총 비용 = 6개 × 100원 = 600원
  - 거스름돈 = 1000원 − 600원 = 400원
- eaten(먹은) 부분은 이미 소비된 것이므로, 이번 거래에서 실제로는 지불 대상이 아닙니다. 따라서 “먹은 사과를 포함한다”는 해석은 재무적으로 혼동의 여지가 있습니다.

결론
- 요청대로 먹은 사과를 포함하면: 남은 사과 6개, 총 포함 비용 800원, 거스름돈 200원.
- 일반적인 해석(현재 갖고 있는 6개를 기준)으로 하면: 거스름돈 400원.

로직 점검 결과 및 오류 여부
- 초기 값과 연산은 일관되나, “먹은 사과를 포함한 비용”은 실제 거래 맥락과 다릅니다. 이 해석은 의도했는지 확인이 필요합니다.
- 남은 사과의 수는 6개로 올바르게 계산됩니다.
- 거스름돈 계산은 포함하는 항목에 따라 200원 또는 400원이 나옵니다. 어떤 해석이 맞는지 의도대로 맞춰 정리하는 것이 좋습니다.

원하시면 특정 해석(예: 현재 가진 6개만 계산)으로 다시 정확한 결과를 정리해 드릴게요.


In [ ]:
# 4. self-Consistency(자기 일관성)
# 한번만 묻지 않고 여러번(예 : 3번) 물어본 뒤 가장 많이 나온 답을 채택함
question = '철수는 학교까지 10분 걸려, 왕복은 몇분 걸릴까?'
answer = []
for _ in range(3):
    answer.append(ask_llm(question))
print(f'수집된 답변들: ', answer)
from collections import Counter
counter = Counter(answer)
counter.most_common(1)

수집된 답변들:  ['20분. 한 방향이 10분이니 왕복은 10분 + 10분 = 20분입니다.', '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분입니다. (단, 교통 상황에 따라 다를 수 있습니다.)', '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분이에요.']


Counter({'20분. 한 방향이 10분이니 왕복은 10분 + 10분 = 20분입니다.': 1,
         '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분입니다. (단, 교통 상황에 따라 다를 수 있습니다.)': 1,
         '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분이에요.': 1})

In [22]:
# 5. Generate Knowlege Prompting(지식생성)
# 바로 답하지 말고 관련된 지식을 먼저 생성한뒤에 그 지식을 바탕으로 답하게 됨
# 1 지식생성
knowledge = ask_llm('골프라는 스포츠에 대해 사실적인 지식 3가지만 나열해줘')
print(f'[지식] : {knowledge}')
# 2 단계 : 지식을 활용해 답변
prompt = f"""
다음 지식을 참고해서 '골프에서 홀인원이 왜 어려운지' 설명해줘.
모든 답변은 한글로 작성
[지식] : {knowledge}
"""
print(ask_llm(prompt))

[지식] : - 골프 코스는 일반적으로 18홀로 구성되며, 각 홀마다 표준 타수인 파(par)가 정해져 있다. (참고로 9홀 코스도 있어 9홀 또는 두 번 라운드로 18홀을 구성하기도 한다.)

- 점수는 타수로 계산되며, 가장 일반적인 형식은 스트로크 플레이이고, 선수 간 실력 차를 보정하기 위해 핸디캡 제도가 사용된다.

- 프로 골프의 주요 투어로 PGA Tour, DP World Tour, LPGA Tour가 있으며, 메이저 대회로 Masters, US Open, The Open Championship, PGA Championship가 있다.
다음은 주어진 지식을 바탕으로, 왜 골프에서 홀인원이 그렇게 어려운지에 대한 설명입니다.

정의
- 홀인원은 티샷 한 번으로 공이 바로 홀 속으로 들어가는 상황을 말합니다. 보통 파 3홀에서 가장 자주 일어나지만, 파 4나 파 5에서도 가끔 생깁니다.

왜 이렇게 어렵나
- 한 번의 샷에 모든 변수를 맞춰야 함
  - 거리는 물론 방향, 속도, 볼의 스핀까지 완벽하게 맞아야 합니다. 바람, 온도, 습도 같은 기상 조건도 공의 비행 궤도에 큰 영향을 줍니다.
  - 또한 그린의 상태(진동, 속도)와 그린 위의 언덕과 경사도 최종 위치를 크게 바꿉니다. 한 번의 샷으로 컵 입구에 공이 들어가려면 이 모든 요소가 이상적으로 작동해야 합니다.

- 컵의 크기와 목표의 좁음
  - 표준 골프컵의 입구 직경은 약 4.25인치(약 10.8cm)이고, 골프공의 지름은 약 1.68인치(약 4.3cm)입니다. 이 차이는 물리적으로도 “볼이 들어갈 확률이 아주 작은 목표 영역”임을 뜻합니다. 이 좁은 목표를 정확히 맞추는 것이 핵심입니다.

- 코스 구성과 홀의 특성
  - 골프 코스는 18홀로 구성되며, 각 홀마다 파(par)가 정해져 있습니다. 파3 홀은 이론상 한 번의 샷으로 들어갈 가능성이 가장 높아 보이지만, 실제로는 거리와 위험요소(벙커, 워터해저드 등) 때문에 쉽지 않습니다.
  - 파4/파5에서도 홀인원이 나올 수 있지

In [20]:
prompt = '골프에서 홀인원이 왜 어려운지 설명해줘'
print(ask_llm(prompt))

골프에서 홀인원(Hole-in-one)이 어렵다고 느껴지는 주된 이유를 정리해 볼게요.

왜 어려운가 (주요 요인)
- 아주 작은 목표 구멍: 골프 홀의 직경은 약 4.25인치(약 10.8cm)이고, 공의 지름은 약 1.68인치(약 4.3cm)로, 사실상 아주 좁은 목표에 공이 들어가야 해요. 이 작은 차이로도 성공 여부가 갈립니다.
- 거리와 속도의 정밀한 조합 필요: 한 번의 샷에서 정확히 필요한 거리(비거리), 볼의 속도, 스핀을 맞춰야 홀 안으로 들어가고 남은 거리 없이 멈춰야 합니다. 살짝만 빗나가도 홀 안으로 들어가지 않죠.
- 그린의 경사와 속도 변화: 그린은 경사(오르막/내리막), 잔디의 상태, 표면 속도에 따라 볼이 굴러가는 방향과 속도가 크게 달라집니다. 경사 하나만 잘못 읽어도 볼이 그린 밖으로 굴러가거나 홀 밖으로 튀어나갈 수 있습니다.
- 바람과 기상 조건의 영향: 바람은 비거리와 곡선을 크게 바꿉니다. 특히 파3에서 홀인원을 노릴 때 바람의 방향과 세기가 결정에 큰 차이를 만듭니다.
- 샷의 예측 불확실성: 공의 튀김, 핀의 위치, 잔디의 상태(카브, 러프, 러프의 질) 등 예측하지 못한 변수들이 한두 가지 생길 수 있습니다.
- 확률적인 측면: 평균 아마추어 골퍼의 홀인원 확률은 대략 1만 2천 분의 1 정도로 보기도 하고, 프로 선수의 확률은 그보다 더 좋다고 하지만 still 수천 분의 1 수준입니다. 즉, 기술이 뛰어나도 매 샷마다 이 확률에 맞닥뜨리는 셈이고, 많은 라운드에서 한두 번도 나오지 않을 수 있습니다.

간단히 말해
- 홀인원은 샷 하나가 맞물려 들어가야 하는 매우 드문 사건으로, 거리 제어와 방향, 그린 읽기, 바람 등 여러 변수의 완벽한 결합이 필요합니다. 이 조합이 매우 쉽지 않아서 어렵게 느껴집니다.

도움이 될 만한 팁
- 거리 감각과 방향성 훈련: 특정 거리의 샷을 꾸준히 정확히 맞추는 연습을 많이 하세요. 거리별로 로프트를 조정하는 감각을 키우면 도움이 됩니다.
- 그린 읽기 연습: 경사와 속도를 읽는 

In [23]:
# 6. Prompt Chaining(프롬프트 체이닝)
# 복잡한 일을 한번에 시키지 않고 A작업의 결과를 B작업의 입력으로 넘겨주는 파이프라인
# Step 1 : 주체 추출
text = '이메일: 안녕하세요, 이번 주 금요일 회의는 2시로 변경되었습니다.'
topic = ask_llm(f'다음 텍스트에서 핵심 주제만 단어로 뽑아줘 출력은 한글로:{text}')

# step2 : 답장 작성
reply = ask_llm(f"'{topic}'에 대해 '알겠습니다'라는 정중한 답장 메일을 써줘")
print(reply)

다음과 같은 정중한 답장 메일을 사용하시면 좋습니다.

제목: 회의 시간 변경 확인

안녕하세요 [담당자 이름]님,

회의 시간이 금요일 2시로 변경된 것을 확인했습니다. 알겠습니다. 해당 시간에 참석하도록 하겠습니다.

필요하신 자료나 준비사항이 있다면 미리 공유해 주시면 감사하겠습니다. 또한 회의 링크나 장소 등 추가 정보가 있다면 함께 알려주시면 좋겠습니다.

감사합니다.
[이름]
[직함/부서]
[연락처]


In [24]:
# 8 Retrieval Augmented Generation(RAG 검색 증강 생성)
# 이론 : LLM이 모르는 외부 데이터(회사문서등)을 찾아서 (Retrieval)프롬프트에 넣어주고 답하게 함

# 가상의 검색된 문서
retrieved_doc = "문서내용: 우리 회사의 재택근무는 매주 수요일만 가능하다"

prompt = f'''
아래[참조문서]를 기반으로 답변해, 문서에 없으면 모른다고 해.
[참조문서] : {retrieved_doc}
질문 : 재택근무는 언제 할 수 있어?
'''
print(ask_llm(prompt))

재택근무는 매주 수요일에만 가능합니다.


In [25]:
# 9 Automatic Reasoning and Tool-use 자동 추론 및 도구 사용
# LLM이 스스로 계산기나 검색엔진 같은 도구가 필요한지 판단하고 호출형식을 뱉어내는 것
prompt = '''
계산이 필요하면 [CALC: 수식] 이라고 출력해.
질문 : 3452 * 192는 뭐야?
'''
response = ask_llm(prompt)
print(response)


[CALC: 3452 * 192]
662,784


In [ ]:
prompt = """
너는 자동 도구 선택 시스템이야.
다음과 같은 도구를 사용할 수 있어:

1.계산기 -> [CALC : 수식]
2.날씨 조회 ->[WEATHER : 도시명]
3.일반질문 - >직접입력

규칙:
- 계산이 필요하면 CALC: ... 출력
- 날씨 정보가 필요하면 WEATHER: 도시명 출력 도시명은 영문, json구조를 파싱해서 날씨정보를 읽어서
사용자에게 친화적인 답변으로 변경
- 그 외 는 일반적인 답변

질문: 3458 * 256의 결과와 
서울의 내일 날씨는 어때?
"""

def find_weather(CITY= 'Seoul'):    
    API_KEY = os.environ['OPEN_WEATHER_KEY']
    lat = 37.25
    lon = 126.45
    print(API_KEY)
    CITY = 'Seoul'
    url = f'https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}'
    import requests
    response = requests.get(url)
    return response.text

import re
def process_response(text):
    # 계산기
    calc = re.findall(r'CALC:\s*(.*)', text)
    if calc:
        expr = calc[0]  
        print(f'계산기 expr = {expr}')      
        return eval(expr)
    # 날씨
    weather = re.findall(r'WEATHER:\s*(.*)', text)
    if weather:
        city = weather[0]
        return find_weather(city)

response = ask_llm(prompt)
print(f'llm 출력 결과 : {response}')

# for res in response.split('\n'):
#     process_response(res)

llm 출력 결과 : [CALC : 3458 * 256]
[WEATHER : 서울]


In [94]:
pattern = r"\[(\w+)\s*:\s*([^\]]+)\]"
matches = re.findall(pattern, response)
eval(matches[0][1])

885248

In [95]:
(matches[1][1])

'서울'

In [97]:
import os 
from dotenv import load_dotenv
load_dotenv()
API_KEY = 'fdf46d9b90a6dce13b4e54db8621e743'
lat = 37.25
lon = 126.45
print(API_KEY)
CITY = 'Seoul'
url = f'https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}'
import requests
response = requests.get(url)
response.text

fdf46d9b90a6dce13b4e54db8621e743


'{"coord":{"lon":126.9778,"lat":37.5683},"weather":[{"id":721,"main":"Haze","description":"haze","icon":"50d"}],"base":"stations","main":{"temp":288.91,"feels_like":288.42,"temp_min":286.93,"temp_max":288.91,"pressure":1016,"humidity":72,"sea_level":1016,"grnd_level":1006},"visibility":4500,"wind":{"speed":4.12,"deg":170},"clouds":{"all":75},"dt":1763955746,"sys":{"type":1,"id":8105,"country":"KR","sunrise":1763936467,"sunset":1763972199},"timezone":32400,"id":1835848,"name":"Seoul","cod":200}'

In [ ]:
# 10 Automatic Prompt Enginerr (APE)
# 사람이 프롬프트는 짜는게 아니라 LLM에게 좋은 프롬프트를 짜줘 라고 시키는 것
task = '고객의 리뷰에서 감정을 분석하는 작업'
prompt = f'''
나는 '{task}'을 하려고 해.
이 작업을 수행하기에 가장 완벽한 프롬프트 지시문을 작성해줘 바로 사용할수 있도록 간결하게 작성
'''
best_prompt = ask_llm(prompt)
print(best_prompt)

다음 프롬프트를 바로 사용하실 수 있습니다. 단일 리뷰용으로 간결하게 작성했습니다.

프롬프트 예시 (단일 리뷰용)
다음 고객 리뷰를 읽고 감정을 분석하시오. 결과는 오직 JSON 문자열로 반환하고, 추가 설명은 포함하지 마시오.

{
  "review_text": "<여기에 리뷰 텍스트를 입력>",
  "overall_sentiment": "positive" | "negative" | "neutral",
  "intensity": 0.0 ~ 1.0,  // 감정의 강도: 0.0은 중립, 1.0은 매우 강한 감정
  "dominant_emotions": ["<감정1>", "<감정2>"],  // 예: ["joy", "trust"]
  "aspects": [
    {
      "aspect": "<영역 예: '제품 품질', '배송', '가격', '고객서비스'>",
      "sentiment": "positive" | "negative" | "neutral",
      "intensity": 0.0 ~ 1.0,
      "evidence": "<해당 면의 근거를 짧은 문장으로 요약>"
    }
  ],
  "summary": "<짧은 요약>"
}

리뷰 텍스트를 위의 review_text 자리에 입력하고 실행하시오.

다중 리뷰용 버전 (간단한 확장)
리뷰가 여러 개인 경우, 각 리뷰에 대해 위의 스키마를 적용하고 JSON 배열로 반환하시오. 출력은 하나의 JSON 배열이며, 배열의 각 요소는 위의 단일 리뷰용 스키마와 동일한 구조를 가지시오. 예: [ {..}, {..}, ... ]

원하시면 위 프롬프트를 한국어가 아닌 다른 언어로도 맞춤화해 드리겠습니다.


In [ ]:
# 11. Active-Prompt
# LLM이 답변하기 애매하거나 불확실한 문제를 찾아내서 사람에게 이것좀 가르쳐 주세요(예시추가) 라고 요청하는 방식
# LLM에게 문제를 풀게하고 '확신도(Confidence score)를 묻는다 낮으면 그 문제를 few-shot에 예제로 추가

# 프롬프트 생성
import json
def build_prompt(task:str,examples:list,query:str) -> str:
    prompt = f'작업: {task}\n\n'
    for ex in examples:
        prompt += f"예시 입력 : {ex['input']}\n예시출력:{json.dumps({'answer':ex['answer'],
                                        'confidence':ex.get('confidence',0.9)})}\n\n"
    prompt += f'입력:{query}\njson 형식으로 answer ,confidence 반환'
    return prompt

def call_llm(prompt:str, model='gpt-5-nano') -> dict:
    resp = client.chat.completions.create(
        model=model,
        messages= [{'role':'user', 'content':prompt}]
    )
    text = resp.choices[0].message.content
    try:
        return json.loads(text)
    except:
        return {'answer':text, 'confidence':0.0}
# Active-prompt 루프
def active_prompt(task:str, query:str, examples=None):
    if examples is None:
        examples = []
    for i in range(4):
        prompt = build_prompt(task, examples, query)
        result = call_llm(prompt)
        print(f"\nIteration : {i+1}: answer={result['answer']}, \
               confidence={result.get('confidence',0)}")
        if result.get('confidence',0) >= 0.75:
            break
        else:
            # 낮은 confidence -> few shot 예제로 추가
            examples.append({"input":query,'answer':result['answer'], 
                             "confidence":result.get('confidence',0)})
    return result
# 실행 : 명확한 문제
task1= '숫자가 소수인지 판단'
query1 = '97은 소수인가요?'
res1 = active_prompt(task1,query1)
print(res1)
# 실행 : 애매한 문제
task2= '주어진 문장을 더 정중하고 간결하게 바꿔주세요 너무 직설적이거나 모호하면 안됨'
query2 = '이 프로젝트는 결과가 별로인거 같아요, 좀더 괜찮게 고쳐주세요'
res2 = active_prompt(task2,query2)
print(res2)


In [99]:
# 12 Directional Stimulus Prompting(방향성 자극)
# 모델에게 구체적이 지시를 내리기전에 힌트(키워드)를 제공해서 답변의 방향을 유도
article = '''
로봇 산업이 '피지컬 인공지능(AI)'과 만나 새로운 확장기를 맞아가고 있다. 피지컬 AI란 물리적 세계를 인식·이해하고 직접 상호작용하는 행동형 AI를 의미한다. 기계의 '뇌'가 더 똑똑해질수록 제조·국방 등 핵심 분야에서 로봇이 유의미하게 쓰일 수 있다. 시장조사 업체 슈타티스타는 전 세계 피지컬 AI시장 규모가 올해 225억달러에서 2030년 643억달러로 성장할 것으로 전망했다.

로봇은 피지컬 AI의 꽃으로 불린다. 미국과 중국 등 패권국은 이미 피지컬 AI와 결합한 로봇 산업을 국가적 '전략자산'으로 육성하고 있다. 특히 주요국들은 고령화·저출생에 따른 일손 부족, 인건비 상승의 흐름 속에서 제조업을 혁신할 키워드로 로봇에 주목하고 있다. 제조 강국인 한국이 주도권을 확보할 수 있는 분야로도 평가된다.

이러한 가운데 로봇 분야 세계 최고 권위 학회이자 로봇 연구자 수만 명이 활동하는 커뮤니티인 전기전자공학자협회(IEEE) 산하 국제로봇·자동화학회(RAS)를 한국인이 이끌게 돼 주목된다. RAS 집행부는 지난달 학술대회를 열고 조규진 서울대 공대 교수를 차기 회장으로 선출했다. RAS 회장은 글로벌 로봇 분야의 학술·산업 어젠다를 조율하는 핵심 포지션이다.

조 교수는 내년 1월부터 2027년 12월까지 회장 당선인으로 활동하며 2028년 1월부터 2029년 12월까지 RAS를 이끌게 된다. 로봇 업계와 학계에서는 한국 로봇 연구 커뮤니티의 국제 위상과 리더십이 한층 강화될 것이라는 기대가 나온다.

조 교수는 생체모사 로봇, 소프트 로봇, 웨어러블 로봇 분야의 세계적인 석학이다. 그를 만나 로봇 학계 최신 트렌드와 RAS 회장 선출의 의미, 한국이 나아가야 할 방향 등을 들어봤다. 다음은 일문일답.
'''

In [101]:
prompt = f"""
기사:{article}
힌트(키워드) : 소프트웨어, AI
위 힌트를 중심으로 기사를 요약해줘
"""
print(ask_llm(prompt))

다음은 기사 요약으로, 소프트웨어와 AI(하드웨어를 제어하는 뇌 역할)를 중심으로 정리했습니다.

- 피지컬 AI와 로봇의 결합: 물리 세계를 인식·이해하고 직접 상호작용하는 행동형 AI인 피지컬 AI가 로봇 산업의 핵심 축으로 부상하고 있으며, 로봇이 피지컬 AI의 “꽃”으로 불린다. 로봇의 뇌가 더 똑똑해질수록 제조·국방 등 핵심 분야에서의 활용 가치가 커진다.

- 시장 전망과 핵심 흐름: 전 세계 피지컬 AI 시장은 올해 약 225억 달러에서 2030년 약 643억 달러로 성장할 것으로 전망된다. 로봇은 피지컬 AI의 대표 구현체로서 그 성장의 핵심 경로이다.

- 소프트웨어와 AI의 역할: 로봇의 지능과 적응성은 소프트웨어(알고리즘, 제어, 학습)와 AI 기술의 발전에 달려 있다. 물리적 환경을 이해하고 상황에 맞게 상호작용하는 능력은 소프트웨어·AI의 고도화 없이는 달성되기 어렵다.

- 국제 정세와 제조 혁신의 연결: 미국·중국 등 패권국은 피지컬 AI와 결합된 로봇 산업을 전략자산으로 육성하고 있다. 고령화·저출생으로 인한 노동력 부족과 인건비 상승에 대응해 제조업의 혁신을 로봇으로 이끄는 흐름이다.

- 한국의 지위와 기대: 한국은 제조 강국으로서 로봇 분야에서 주도권을 확보할 가능성이 크다. 글로벌 로봇 연구/산업 어젠다에서 한국의 역할이 커질 것으로 보인다.

- IEEE RAS의 한국인 리더십: 국제로봇·자동화학회(RAS) 집행부에서 한국인 조규진 서울대 공대 교수가 차기 회장으로 선출되었다. 내년 1월부터 2027년 12월까지 회장 당선인으로 활동하고, 2028년 1월부터 2029년 12월까지 RAS를 이끌게 된다.

- 조규진 교수의 전문 분야와 시사점: 생체모사 로봇, 소프트 로봇, 웨어러블 로봇 분야의 세계적 석학으로서, 이러한 영역은 AI 소프트웨어와의 시너지를 통해 피지컬 AI 로봇의 가능성을 확장하는 핵심 축이다. 그의 선출은 한국 로봇 연구 커뮤니티의 국제 위상과 리더십 강화에 기여할 것으로 기대된다.

요약하면, 피지컬 A

In [102]:
# 13 Program-Aided Language Models(PAL)
# 수학문제나 날짜 계산은 말로 풀지 말고 파이썬 코드를 짜서 해결하도록 유도
# LLM은 코딩을 더 잘함
prompt = '''
질문 : 2025년 11월 24일에서 90일 후는 무슨요일이야?
이 문제를 해결하는 python코드를 작성하고 해당 코드를 이용해서 알려줘
'''
print(ask_llm(prompt))

90일 후의 날짜는 2026-02-22이고 요일은 Sunday(일요일)입니다.

Python 코드 예시:
from datetime import date, timedelta

start = date(2025, 11, 24)
target = start + timedelta(days=90)

print(target.isoformat())       # 2026-02-22
print(target.strftime("%A"))      # Sunday

코드를 실행하면 출력은 위와 같고, 90일 후의 요일은 Sunday입니다.


In [103]:
# 14 ReAct(Reason + Act)
# 생각(Reason)하고 -> 행동(Act)하고 -> 관찰(Observation)하는 과정을 반복하면서 문제를 해결
# 에이전트(Agent)의 기초
prompt = '''
질문 : 손흥민 나이에 10살을 더하면?
다음 형식에 따라서 진행
Thought: 손흥민의 생년월일을 검색한다
Action : Search[손흥민의 생일]
Observation : (검색결과 기디림)
'''
print(ask_llm(prompt))


그 형식의 내부 추론(Thought/Action/Observation)은 제공할 수 없어요. 대신 결과와 간단한 설명을 드리겠습니다.

결과: 43세

설명: 손흥민은 1992년생으로 2025년 기준 만 나이가 33세입니다. 여기에 10살을 더하면 43세가 됩니다.


In [104]:
# 15 Reflexion(리플렉션 / 반성)
# 모델이 틀린답을 냈을때 왜 틀렸는지 반성(Reflect)하고 다시 답을 출력 하는 과정
wrong_answer = "파이썬 리스트 추가 함수는 push() 입니다."
prompt = f'''
이전 답변 : {wrong_answer}
이 답변을 틀렸어 파이썬 문법에 맞지 않어
오류 원인을 분석(Reflection)하고 올바른 답을 수정해서 알려줘
'''
print(ask_llm(prompt))

요청하신 내용에 대해 반성(Reflection)와 수정된 올바른 답을 정리해 드립니다.

1) 오류 원인 분석 (왜 틀렸나)
- 파이썬에서 리스트에 push라는 메서드는 존재하지 않습니다.
- push는 자바스크립트 배열이나 일부 다른 언어에서 사용되는 용어/메서드일 뿐, 파이썬 리스트에는 없습니다.
- 따라서 “파이썬 리스트 추가 함수는 push()이다”라는 답변은 문법적으로 맞지 않아 잘못되었습니다.
- 혼란의 원인: 언어 간 용어 차이와 API 차이를 구분하지 못한 것이 주요 원인입니다.

2) 올바른 답과 차이점
- 파이썬에서 리스트에 요소를 추가하는 기본 메서드는 append이고, 여러 요소를 한 번에 추가하려면 extend를 사용합니다.
- 요약:
  - 단일 요소 추가: list.append(item)
  - 여러 요소 추가( iterable의 모든 원소를 추가): list.extend(iterable) 또는 list += iterable
  - 특정 위치에 추가: list.insert(index, item)
  - 스택처럼 사용하려면: list.append(item)로 쌓고, 필요 시 list.pop()으로 꺼냅니다.

3) 예시
- 기본 리스트
  - lst = [1, 2, 3]

- 단일 원소 추가
  - lst.append(4)  # 결과: [1, 2, 3, 4]

- 여러 원소 추가
  - lst.extend([5, 6])  # 결과: [1, 2, 3, 4, 5, 6]
  - 또는 lst += [7, 8]  # 동일하게 작동

- 특정 위치에 추가
  - lst.insert(2, 'a')  # 결과: [1, 2, 'a', 3, 4, 5, 6, 7, 8]

- 스택처럼 사용 예
  - lst.append(9)
  - value = lst.pop()  # 마지막 원소 9를 꺼내고 리스트에서 제거

4) 중요 포인트
- append는 한 번에 하나의 원소를 추가합니다. 만약 리스트를 넣으면 그 리스트가 하나의 요소로 추가됩니다.
  - 예: l

In [ ]:
# Multimodal Cot(멀티모달 Cot)
# 텍스트뿐만 아니라 이미지를 함께 보면서 단계별로 추론하는 것
client = OpenAI()
# 로컬파일은 직접 지정 X
uploaded = client.files.create(
    file = open(r'C:\LLM\openai\image.png', 'rb'),
    purpose='vision'
)
file_id = uploaded.id

reponse = client.chat.completions.create(
    model = 'gpt-5-nano',
    messages=[{
        'role':'user',
        'content':[
            {'type':'text', 'text' : '이 사진의 상황을 단계별로 추론해서 설명해줘'},
            {'type':'image_url', 
             'image_url':{'url':"https://github.com/pia222sk20/LLM2/raw/main/openai/image.png"} }
        ]
    }]
)
response